# Notebook 04: LLM Generation

**Goal:** Connect to local Ollama (Llama 3 model), construct strict grounding prompt templates, test factual vs. off-topic query generation, and implement streaming response generation.

## Objectives:
1. Connect to Ollama API (`ChatOllama` with model `llama3`)
2. Design strict document-grounding prompt template to prevent hallucination
3. Test factual document query synthesis (2-4 concise sentences)
4. Test off-topic query fallback handling
5. Verify token streaming capabilities and run verification tests

In [1]:
# Cell 1: Import dependencies & initialize ChatOllama
import os
import sys
from typing import List, Dict, Any

project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import dotenv

dotenv.load_dotenv(os.path.join(project_root, ".env"))
ollama_host = os.getenv("OLLAMA_HOST", "http://localhost:11434")
ollama_model = os.getenv("OLLAMA_MODEL", "llama3")

print(f"Connecting to ChatOllama at {ollama_host} with model '{ollama_model}'...")
llm = ChatOllama(
    model=ollama_model,
    base_url=ollama_host,
    temperature=0.0,
    num_predict=256
)
print("ChatOllama client successfully initialized.")

Connecting to ChatOllama at http://localhost:11434 with model 'llama3'...
ChatOllama client successfully initialized.


In [2]:
# Cell 2: Prompt Template & LCEL Chain Construction
SYSTEM_PROMPT_TEMPLATE = """You are a precise document analysis assistant.
Answer the user's question using ONLY the provided context chunks below.
Strict Rules:
1. Base your answer strictly on the facts present in the context.
2. Do NOT use external knowledge or invent facts.
3. If the answer cannot be determined from the context, state: 'I could not find that information in the provided document.'
4. Keep the answer concise and direct (2 to 4 sentences).

Context Chunks:
{context}

Question: {question}
Answer:
"""

prompt = ChatPromptTemplate.from_template(SYSTEM_PROMPT_TEMPLATE)
chain = prompt | llm | StrOutputParser()
print("LCEL Grounded Generation Chain built successfully.")

LCEL Grounded Generation Chain built successfully.


In [3]:
# Cell 3: Factual Question Test
sample_context = """
[Chunk chunk_003 | Section: 2. Hierarchical Retrieval Architecture]
The cross-encoder performs deep cross-attention between the query and candidate passages to compute precise semantic similarity scores. The top 3 reranked chunks are subsequently formatted into a structured context window and forwarded to the local language model for answer synthesis.

[Chunk chunk_006 | Section: 4. System Performance and Resource Requirements]
The average CPU latency for BM25 retrieval across a 50-page document is under 15 milliseconds. Cross-encoder reranking adds approximately 150 to 300 milliseconds on standard x86 and ARM processors.
"""

factual_question = "How much latency does cross-encoder reranking add on standard processors?"
response_factual = chain.invoke({"context": sample_context, "question": factual_question})
print(f"Question: '{factual_question}'")
print(f"Generated Answer:\n{response_factual.strip()}")

Question: 'How much latency does cross-encoder reranking add on standard processors?'
Generated Answer:
According to the context, cross-encoder reranking adds approximately 150 to 300 milliseconds on standard x86 and ARM processors.


In [4]:
# Cell 4: Off-Topic Question Test (Hallucination Prevention)
offtopic_question = "What is the population of Tokyo in 2026?"
response_offtopic = chain.invoke({"context": sample_context, "question": offtopic_question})
print(f"Off-topic Question: '{offtopic_question}'")
print(f"Generated Answer:\n{response_offtopic.strip()}")

Off-topic Question: 'What is the population of Tokyo in 2026?'
Generated Answer:
I could not find that information in the provided document. The context only discusses hierarchical retrieval architecture, system performance, and resource requirements, but does not mention the population of Tokyo or any other city, including Tokyo's population in 2026.


In [5]:
# Cell 5: Streaming Response Test
streaming_question = "What is the average CPU latency for BM25 retrieval?"
print(f"Streaming test for question: '{streaming_question}'\n---")
stream_chunks = []
for chunk in chain.stream({"context": sample_context, "question": streaming_question}):
    stream_chunks.append(chunk)
    print(chunk, end="", flush=True)
print("\n---\nStreaming completed successfully.")

Streaming test for question: 'What is the average CPU latency for BM25 retrieval?'
---


According

 to

 Chunk

 chunk

...

 under

15

 milliseconds

.


---
Streaming completed successfully.


In [6]:
# Cell 6: Verification Assertions for Notebook 04
print("Running verification assertions on LLM Generation...")
# 1. Factual answer must contain key fact from context
assert len(response_factual.strip()) > 0, "Verification Failed: Factual response is empty!"
assert any(val in response_factual.lower() for val in ["150", "300", "milliseconds", "ms"]), "Verification Failed: Factual answer missing latency numbers!"

# 2. Off-topic answer must trigger fallback
assert "could not find" in response_offtopic.lower() or "not mentioned" in response_offtopic.lower() or "not provided" in response_offtopic.lower() or "unable" in response_offtopic.lower(), "Verification Failed: Off-topic query did not trigger fallback statement!"

# 3. Streaming chunks must concatenate to valid text
full_streamed_text = "".join(stream_chunks)
assert len(full_streamed_text) > 0, "Verification Failed: Streamed response was empty!"

print("✅ ALL VERIFICATION TESTS PASSED FOR NOTEBOOK 04!")

Running verification assertions on LLM Generation...
✅ ALL VERIFICATION TESTS PASSED FOR NOTEBOOK 04!
